In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

In [2]:
DATA_PATH = Path("../Data/Cleaned/Nepal_Online_Payment_System.csv")

df = pd.read_csv(DATA_PATH)

In [4]:
df.shape
df.head()

,period_end,bs_year,bs_month,fiscal_year_bs,instrument,category,txn_count,txn_amount_npr_million,avg_txn_value_npr,is_flagged,flag_reason
0,2023-07,2080,Asar,2079/80,RTGS,Interbank / large value,76307,2983930,39104276.15,False,NaN
1,2023-07,2080,Asar,2079/80,ATM Cash Withdrawal,Cash,11042117,86964,7875.66,False,NaN
2,2023-07,2080,Asar,2079/80,ECC (Cheque Clearing),Clearing,1336586,718755,537754.40,False,NaN
3,2023-07,2080,Asar,2079/80,IPS,Interbank / retail,1808046,323816,179097.21,False,NaN
4,2023-07,2080,Asar,2079/80,Faster Payment Systems,Instant payments,9783842,321519,32862.24,False,NaN


In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 645 entries, 0 to 644
Data columns (total 11 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   period_end              645 non-null    str    
 1   bs_year                 645 non-null    int64  
 2   bs_month                645 non-null    str    
 3   fiscal_year_bs          645 non-null    str    
 4   instrument              645 non-null    str    
 5   category                645 non-null    str    
 6   txn_count               645 non-null    int64  
 7   txn_amount_npr_million  645 non-null    int64  
 8   avg_txn_value_npr       645 non-null    float64
 9   is_flagged              645 non-null    bool   
 10  flag_reason             12 non-null     str    
dtypes: bool(1), float64(1), int64(3), str(6)
memory usage: 51.1 KB


In [6]:
df.isnull().sum()

period_end                  0
bs_year                     0
bs_month                    0
fiscal_year_bs              0
instrument                  0
category                    0
txn_count                   0
txn_amount_npr_million      0
avg_txn_value_npr           0
is_flagged                  0
flag_reason               633
dtype: int64

In [7]:
df.duplicated().sum()

np.int64(0)

In [9]:
pd.set_option("display.max_colwidth", None)

display(
    df[df["is_flagged"] == True][
        ["period_end", "instrument", "flag_reason"]
    ]
)

,period_end,instrument,flag_reason
61,2023-11,IPS,"Source revision: Kartik PDF reported amount 133,124; Mangsir PDF revised to 133,089 (revised value used)"
66,2023-11,Mobile Banking,"Source revision: Kartik PDF reported count 25,649,080 / amount 218,309; Mangsir PDF revised to 26,981,617 / 228,328 (revised value used)"
103,2024-02,IPS,IPS count spike (>=4M vs ~1-3M typical); likely bulk/institutional batches
175,2024-07,IPS,IPS count spike (>=4M vs ~1-3M typical); likely bulk/institutional batches
223,2024-10,IPS,IPS count spike (>=4M vs ~1-3M typical); likely bulk/institutional batches
376,2025-07,IPS,IPS count spike (>=4M vs ~1-3M typical); likely bulk/institutional batches
427,2025-10,IPS,IPS count spike (>=4M vs ~1-3M typical); likely bulk/institutional batches
478,2026-01,IPS,IPS count spike (>=4M vs ~1-3M typical); likely bulk/institutional batches
495,2026-02,IPS,IPS count spike (>=4M vs ~1-3M typical); likely bulk/institutional batches
580,2026-07,IPS,IPS count spike (>=4M vs ~1-3M typical); likely bulk/institutional batches


In [10]:
df["period_end"] = pd.to_datetime(df["period_end"])

df["period_end"].dtype

dtype('<M8[us]')

In [11]:
print("First reporting month:", df["period_end"].min())
print("Last reporting month:", df["period_end"].max())

print("Total reporting months:", df["period_end"].nunique())

First reporting month: 2023-07-01 00:00:00
Last reporting month: 2026-10-01 00:00:00
Total reporting months: 40


In [12]:
available_months = sorted(df["period_end"].unique())

available_months

[Timestamp('2023-07-01 00:00:00'),
 Timestamp('2023-08-01 00:00:00'),
 Timestamp('2023-09-01 00:00:00'),
 Timestamp('2023-10-01 00:00:00'),
 Timestamp('2023-11-01 00:00:00'),
 Timestamp('2023-12-01 00:00:00'),
 Timestamp('2024-01-01 00:00:00'),
 Timestamp('2024-02-01 00:00:00'),
 Timestamp('2024-03-01 00:00:00'),
 Timestamp('2024-04-01 00:00:00'),
 Timestamp('2024-05-01 00:00:00'),
 Timestamp('2024-06-01 00:00:00'),
 Timestamp('2024-07-01 00:00:00'),
 Timestamp('2024-08-01 00:00:00'),
 Timestamp('2024-09-01 00:00:00'),
 Timestamp('2024-10-01 00:00:00'),
 Timestamp('2024-11-01 00:00:00'),
 Timestamp('2024-12-01 00:00:00'),
 Timestamp('2025-01-01 00:00:00'),
 Timestamp('2025-02-01 00:00:00'),
 Timestamp('2025-03-01 00:00:00'),
 Timestamp('2025-04-01 00:00:00'),
 Timestamp('2025-05-01 00:00:00'),
 Timestamp('2025-06-01 00:00:00'),
 Timestamp('2025-07-01 00:00:00'),
 Timestamp('2025-08-01 00:00:00'),
 Timestamp('2025-09-01 00:00:00'),
 Timestamp('2025-10-01 00:00:00'),
 Timestamp('2025-11-

In [14]:
all_months = pd.date_range(
    start=df["period_end"].min(),
    end=df["period_end"].max(),
    freq="MS"
)

missing_months = all_months.difference(
    df["period_end"].drop_duplicates()
)

print("Missing months:", len(missing_months))
print(missing_months)

Missing months: 0
DatetimeIndex([], dtype='datetime64[us]', freq='MS')


In [15]:
monthly_coverage = (
    df.groupby("period_end")["instrument"]
    .nunique()
    .reset_index(name="instruments_reported")
)

monthly_coverage

,period_end,instruments_reported
0,2023-07-01,16
1,2023-08-01,14
2,2023-09-01,14
3,2023-10-01,14
4,2023-11-01,14
5,2023-12-01,14
6,2024-01-01,14
7,2024-02-01,14
8,2024-03-01,14
9,2024-04-01,14


In [16]:
monthly_coverage["instruments_reported"].value_counts().sort_index()

instruments_reported
14     9
15     2
16     4
17    25
Name: count, dtype: int64

In [17]:
availability_matrix = (
    df.pivot_table(
        index="period_end",
        columns="instrument",
        values="txn_count",
        aggfunc="count"
    )
    .notna()
    .astype(int)
)

availability_matrix

instrument,ATM Cash Withdrawal,Branchless Banking,Credit Cards,Cross Border QR Acquiring,Debit Cards,E-Commerce (card-based),ECC (Cheque Clearing),Faster Payment Systems,IPS,Internet Banking,Mobile Banking,Other Retail Payments,Point of Sales (POS),Prepaid Cards,QR-Based Payments,RTGS,Wallet
period_end,,,,,,,,,,,,,,,,,
2023-07-01,1,1,1,0,1,1,1,1,1,1,1,1,1,1,1,1,1
2023-08-01,1,1,1,0,1,1,1,0,1,1,1,0,1,1,1,1,1
2023-09-01,1,1,1,0,1,1,1,0,1,1,1,0,1,1,1,1,1
2023-10-01,1,1,1,0,1,1,1,0,1,1,1,0,1,1,1,1,1
2023-11-01,1,1,1,0,1,1,1,0,1,1,1,0,1,1,1,1,1
2023-12-01,1,1,1,0,1,1,1,0,1,1,1,0,1,1,1,1,1
2024-01-01,1,1,1,0,1,1,1,0,1,1,1,0,1,1,1,1,1
2024-02-01,1,1,1,0,1,1,1,0,1,1,1,0,1,1,1,1,1
2024-03-01,1,1,1,0,1,1,1,0,1,1,1,0,1,1,1,1,1


In [18]:
availability_matrix.sum().sort_values()

instrument
Cross Border QR Acquiring    25
Other Retail Payments        29
Faster Payment Systems       31
Branchless Banking           40
Credit Cards                 40
Debit Cards                  40
E-Commerce (card-based)      40
ATM Cash Withdrawal          40
ECC (Cheque Clearing)        40
IPS                          40
Internet Banking             40
Mobile Banking               40
Point of Sales (POS)         40
Prepaid Cards                40
QR-Based Payments            40
RTGS                         40
Wallet                       40
dtype: int64

In [19]:
missing_instruments = (
    availability_matrix
    .eq(0)
    .stack()
    .reset_index()
)

missing_instruments.columns = [
    "period_end",
    "instrument",
    "not_reported"
]

missing_instruments = missing_instruments[
    missing_instruments["not_reported"]
]

missing_instruments

,period_end,instrument,not_reported
3,2023-07-01,Cross Border QR Acquiring,True
20,2023-08-01,Cross Border QR Acquiring,True
24,2023-08-01,Faster Payment Systems,True
28,2023-08-01,Other Retail Payments,True
37,2023-09-01,Cross Border QR Acquiring,True
41,2023-09-01,Faster Payment Systems,True
45,2023-09-01,Other Retail Payments,True
54,2023-10-01,Cross Border QR Acquiring,True
58,2023-10-01,Faster Payment Systems,True
62,2023-10-01,Other Retail Payments,True


In [20]:
missing_instruments.groupby("instrument").size()

instrument
Cross Border QR Acquiring    15
Faster Payment Systems        9
Other Retail Payments        11
dtype: int64